# Joi — Schema Validation for JavaScript & Node.js

Joi is a schema description and data validation library. You describe the shape data *should* have, then hand Joi the actual data and it tells you whether it conforms — and if not, exactly where and why.

Most commonly used at the boundary of an application: validating HTTP request bodies, query params, config files, and environment variables before that data reaches business logic.

---

## Install

```bash
npm install joi
```

> **Package name history:** Joi lived at `@hapi/joi` for versions 16–17.0. From v17 onward it moved back to plain `joi`. If you find a tutorial using `@hapi/joi`, it's stale — use `joi`.

```js
// ESM
import Joi from 'joi';

// CommonJS
const Joi = require('joi');
```

---

## Basic Example

```js
import Joi from 'joi';

const schema = Joi.object({
  username: Joi.string().alphanum().min(3).max(30).required(),
  email: Joi.string().email().required(),
  age: Joi.number().integer().min(18),
});

const { error, value } = schema.validate({
  username: 'danilo',
  email: 'danilo@example.com',
  age: 28,
});

if (error) {
  console.error(error.details);
} else {
  console.log(value); // the validated + coerced object
}
```

Two things worth internalizing right away:

1. **`validate()` does not throw.** It returns `{ value, error }`. You must check `error` yourself.
2. **Use the returned `value`, not your original input.** Joi coerces by default, so `"28"` comes back as `28`. Ignoring `value` throws away that work.

---

## The Three Ways to Validate

| Method | Behavior | Use when |
|---|---|---|
| `schema.validate(data, opts)` | Returns `{ value, error }` | Default, sync |
| `await schema.validateAsync(data, opts)` | Resolves to `value`, **throws** `ValidationError` | Async rules (`.external()`), or you prefer try/catch |
| `Joi.attempt(data, schema)` | Returns `value`, **throws** on failure | Fail-fast startup checks (e.g. env config) |

```js
// Config validation at boot — crash loudly if the environment is wrong
const env = Joi.attempt(process.env, envSchema, 'Invalid environment:');
```

---

## Validation Options

Passed as the second argument to `validate()`, or baked in with `schema.options({ ... })`.

| Option | Default | What it does |
|---|---|---|
| `abortEarly` | `true` | Stop at the first error. Set to `false` for forms — you want *all* the errors. |
| `convert` | `true` | Coerce types (`"28"` → `28`, `"true"` → `true`, date strings → `Date`). |
| `allowUnknown` | `false` | Permit keys not in the schema instead of erroring. |
| `stripUnknown` | `false` | Silently delete unknown keys from the output. |
| `presence` | `'optional'` | Global default: `'optional'`, `'required'`, or `'forbidden'`. |
| `context` | `{}` | Extra data available to `.when()` and refs via `$` prefix. |

```js
const { error, value } = schema.validate(req.body, {
  abortEarly: false,
  stripUnknown: true,
});
```

`allowUnknown: true` + `stripUnknown: true` is the usual combination for API request bodies: don't reject the request over a stray field, but don't let it through to your database either.

---

## Reading the Error

`error` is a `ValidationError`. The useful part is `error.details` — an array, one entry per failure:

```js
[
  {
    message: '"email" must be a valid email',
    path: ['email'],
    type: 'string.email',
    context: { label: 'email', key: 'email', value: 'not-an-email' }
  }
]
```

A typical formatter for an API response:

```js
const format = (error) =>
  error.details.map((d) => ({
    field: d.path.join('.'),
    message: d.message,
  }));
```

`type` is the stable, machine-readable identifier (`string.min`, `any.required`, `number.base`). Match on `type`, never on `message`.

---

## Core Types & Common Rules

### String

```js
Joi.string()
  .min(3).max(30)
  .alphanum()                     // letters + digits only
  .pattern(/^[a-z0-9_]+$/)        // custom regex
  .email({ tlds: { allow: false } })
  .uri()
  .uuid()
  .lowercase()                    // coerces when convert: true
  .trim()
  .valid('draft', 'published')    // enum
  .allow('')                      // permit empty string
  .required();
```

> **Gotcha:** `Joi.string().required()` rejects `''` with `string.empty`. If an empty string is legitimate, add `.allow('')`.

> **Gotcha:** `.email()` validates the TLD against a real list by default. In some bundled/browser environments that list isn't available — `{ tlds: { allow: false } }` sidesteps it.

### Number

```js
Joi.number()
  .integer()
  .positive()        // > 0
  .min(0).max(100)
  .precision(2)
  .multiple(5)
  .default(0);
```

### Boolean

```js
Joi.boolean().truthy('yes', '1').falsy('no', '0');
```

### Date

```js
Joi.date()
  .iso()                  // require ISO 8601 format
  .min('1-1-2020')
  .max('now')
  .greater(Joi.ref('startDate'));
```

### Array

```js
Joi.array()
  .items(Joi.string())
  .min(1).max(10)
  .unique()
  .single();              // accept a bare value as a 1-element array
```

### Object

```js
Joi.object({
  id: Joi.string().uuid(),
  meta: Joi.object().unknown(true),   // allow arbitrary keys here only
})
  .or('email', 'phone')        // at least one
  .xor('cardId', 'upiId')      // exactly one
  .and('lat', 'lng')           // all or none
  .with('password', 'confirmPassword')  // if password, then confirm too
  .without('guestId', ['userId']);
```

### Any / Alternatives

```js
Joi.alternatives().try(Joi.string(), Joi.number());

Joi.any().valid('a', 'b');
Joi.any().forbidden();
```

---

## Conditional Validation with `.when()`

The feature that pushes Joi past simple type checking.

```js
const schema = Joi.object({
  accountType: Joi.string().valid('personal', 'business').required(),

  gstNumber: Joi.string().when('accountType', {
    is: 'business',
    then: Joi.required(),
    otherwise: Joi.forbidden(),
  }),
});
```

### References

`Joi.ref()` points at a sibling key. The classic password-confirmation case:

```js
const schema = Joi.object({
  password: Joi.string().min(8).required(),
  confirmPassword: Joi.any()
    .valid(Joi.ref('password'))
    .required()
    .messages({ 'any.only': 'Passwords do not match' }),
});
```

External context is referenced with `$`:

```js
const schema = Joi.object({
  role: Joi.string().when('$isAdmin', {
    is: true,
    then: Joi.valid('admin', 'user'),
    otherwise: Joi.valid('user'),
  }),
});

schema.validate(data, { context: { isAdmin: req.user.isAdmin } });
```

---

## Custom Messages & Labels

Default messages leak your internal key names (`"gstNumber" is required`). Two fixes:

```js
// Rename the key in messages
Joi.string().required().label('GST Number');

// Or override per error type
Joi.string().min(8).required().messages({
  'string.min': 'Password must be at least {#limit} characters',
  'string.empty': 'Password cannot be blank',
  'any.required': 'Password is required',
});
```

Template variables available inside messages: `{#label}`, `{#value}`, `{#limit}`, `{#key}`.

---

## Custom Validators

For logic Joi doesn't ship:

```js
const objectId = Joi.string().custom((value, helpers) => {
  if (!/^[0-9a-fA-F]{24}$/.test(value)) {
    return helpers.error('any.invalid');
  }
  return value;
}, 'MongoDB ObjectId');
```

For async checks (database lookups), use `.external()` — this requires `validateAsync`:

```js
const schema = Joi.object({
  email: Joi.string().email().required(),
}).external(async (value) => {
  const exists = await User.exists({ email: value.email });
  if (exists) throw new Error('Email already registered');
});

await schema.validateAsync(req.body);
```

> Don't reach for `.external()` reflexively. Uniqueness checks at validation time are racy — you still need a unique index in the database.

---

## Express Middleware

A single reusable factory covers most of an API surface:

```js
// middleware/validate.js
export const validate = (schema, property = 'body') => (req, res, next) => {
  const { error, value } = schema.validate(req[property], {
    abortEarly: false,
    stripUnknown: true,
  });

  if (error) {
    return res.status(400).json({
      message: 'Validation failed',
      errors: error.details.map((d) => ({
        field: d.path.join('.'),
        message: d.message,
      })),
    });
  }

  req[property] = value;   // replace with the coerced/stripped version
  next();
};
```

```js
// routes/users.js
router.post('/users', validate(createUserSchema), createUser);
router.get('/users', validate(listUsersSchema, 'query'), listUsers);
```

The `req[property] = value` line is the part people skip. Without it you validate the data and then go on using the unvalidated original.

---

## Environment Config Validation

An underrated use. Fail at boot rather than at 2 AM when a missing variable turns into `undefined`:

```js
const envSchema = Joi.object({
  NODE_ENV: Joi.string().valid('development', 'production', 'test').required(),
  PORT: Joi.number().default(3000),
  MONGO_URI: Joi.string().uri().required(),
  JWT_SECRET: Joi.string().min(32).required(),
}).unknown(true);   // process.env has hundreds of other keys

export const config = Joi.attempt(process.env, envSchema, 'Config error:');
```

---

## Joi and Mongoose

They validate at different layers and it's fine to have both:

- **Joi** validates the *request* — untrusted input, shaped for HTTP, with messages meant for humans.
- **Mongoose** validates the *document* — the last guard before persistence, catching writes that don't come through your routes.

Joi gives you far richer conditional logic and better error output; Mongoose gives you a guarantee that no code path can bypass. Duplicating a few `required` flags across both is a reasonable cost.

---

## Common Pitfalls

| Pitfall | Fix |
|---|---|
| Discarding `value` and using the raw input | Always use the returned `value` |
| Only the first error shows up on a form | `abortEarly: false` |
| Numbers arrive as strings from query params | Leave `convert: true` (the default) — that's what it's for |
| Unknown keys reach the database | `stripUnknown: true` |
| Error messages expose internal field names | `.label()` or `.messages()` |
| Rebuilding the schema on every request | Define schemas once at module scope; they're immutable and reusable |
| Matching on `error.details[0].message` | Match on `.type` instead |

Schemas in Joi are **immutable** — every chained call returns a new schema. `schema.min(3)` on its own line does nothing unless you reassign it.

---

## Joi vs. Zod (brief)

If you're choosing today and working in TypeScript, this comparison usually decides it:

- **Joi** — more mature, more built-in validators, richer conditionals (`.when()`, refs, object relations like `xor`/`with`). Weak TypeScript story: types aren't inferred from schemas.
- **Zod** — TypeScript-first, schemas infer static types automatically (`z.infer<typeof schema>`), smaller API surface.

For plain JavaScript Node backends, Joi remains an excellent choice. For TypeScript, the type inference is a genuine advantage for Zod.

---

## References

- [Joi official documentation](https://joi.dev/) — the API reference is exhaustive; keep it open
- [Joi API — validation options](https://joi.dev/api/)
- [Express](https://expressjs.com/)
- [npm: joi](https://www.npmjs.com/package/joi)